# လူ့ကုတ်ထဲတွင်လက်ဝင်ခြင်း: ကြိုတင်လုပ်ဆောင်ချက် မတားဆီးမှုများ၊ အန္တရာယ်အဆင့်ခွဲခြားခြင်းနှင့် စစ်ဆေးမှတ်တမ်းပြုစုခြင်း

ဒီသင်ခန်းစာ၏ README သည် လူ့ကုတ်ထဲတွင်လက်ဝင်ခြင်းကို အသုံးပြုသူအား `APPROVE` သို့မဟုတ် `REJECT` ဆုံးဖြတ်ရန် agent သည် ပြန်လည် တုံ့ပြန်ချက် ထုတ်ပေးပြီးနောက် လိုက်နာစေသည့် စနစ်တစ်ခုကို မိတ္တူကျယ်ကန့်တည်း ဖော်ပြသည်။ ထိုအစဉ်အလာသည် စတင်ရန်အတွက် သင့်တော်သော်လည်း ထုတ်လုပ်မှု HITL တွင် သုံးသည့် အစိတ်အပိုင်းသုံးခုကို ပိုမိုလိုအပ်သည်။

1. agent သည် အန္တရာယ်ရှိသော လှုပ်ရှားချက် တစ်ခုကို လုပ်ဆောင်ရန်မတိုင်မီ **ကြိုတင်လုပ်ဆောင်ချက် မတားဆီးမှု** တစ်ရပ်ကို ထည့်သွင်း၊ ထိုဆောင်ရွက်မှုသည် ကုန်ကျစရိတ်၊ မပြန်နိုင်ခြင်းနှင့် ထိန်းချုပ်ရသော ကြာမြင့်ချိန်ကို ထိန်းသိမ်းစေသည်။
2. အန္တရာယ်အဆင့်ခွဲခြားမှု၊ အန္တရာယ်နည်းသော လှုပ်ရှားချက်များကို အလိုအလျောက် လုပ်ဆောင်ခြင်း၊ အန္တရာယ်အလယ်အလတ် များကို အစုလိုက်အပြုံလိုက် ခွင့်ပြုကာ၊ အန္တရာယ်အလွန်အမင်းများသာ လူလက်တွင်တားဆီးခြင်း။
3. **စစ်တမ်းမှတ်တမ်းနှင့် ပြန်လည်သုံးသပ်ခြင်း စက်ကွင်း**, မတားဆီးမှုဆုံးဖြတ်ချက်တိုင်းကို JSONL အဖြစ် မှတ်တမ်းတင်ပြီး၊ ပယ်ချခြင်းသည် `Revising...` လို့သာ မထုတ်ပြသပဲ ဖွဲ့စည်းထားသော အကြောင်းပြချက်ဖြင့် agent ကို ပြန်လည်မေးခွန်းတင်ခြင်း။

ဤ notebook သည် `06-system-message-framework.ipynb` အောက်ခံအစိတ်အပိုင်းများကို အသုံးပြု၍ ပြုလုပ်ထားသည်။ `DEMO_MODE = True` တွင် (အင်တာက်တက်ဖြစ်သည့် အထောက်အထားမလိုသော) သို့မဟုတ် `DEMO_MODE = False` ဖြစ်သော အခါ နောက်ဆက်တွဲ `input()` ဆိုရှ်များဖြင့် အပြီးအစီး လည်ပတ်နိုင်သည်။ သတိပြုရန် - DEMO_MODE တွင် တတိယရည်မှန်းချက်၏ ထပ်မံကြိုးစားမှုကို စကရစ်ပစ်ဖြင့်ပြုလုပ်ထား၍ အသေးစိတ်၍ လည်ပတ်မှု မြင်သာစေရန်ဖြစ်သည်။ အမှန်တကယ် ပြန်လည်သုံးသပ်ခြင်းမှ ဖြစ်ပေါ်သော ပြန်လည်ခွဲခြားမှုများအတွက် `DEMO_MODE = False` နှင့် လုပ်ကိုင်သူ လိုအပ်သည်။

**အခြားသင်ခန်းစာများတွင် ကိုင်တွယ်ထားသောအကြောင်းအရာများ (scope အပြင်):** အတည်ပြုခြင်းနှင့် ဝင်ရောက်ခွင့်ထိန်းချုပ်ခြင်း (သင်ခန်းစာ 06 README အန္တရာယ်အချက် 2), စက်မှုပေါ်သုံး middleware (သင်ခန်းစာ 14 MAF နက်ရှိုင်းစွာ လေ့လာခြင်း), မျိုးစုံ agent ဆွေးနွေးပွဲပုံစံများ။


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## ပုံစံ ၁: လုပ်ဆောင်မှုမရှိမီ တံခါးတစ်ချပ်

README ရဲ့ HITL ကိုးကားမှုသည် အေးဂျင့်ကို ပထမဆုံး ခေါ်ပြီးနောက် အသုံးပြုသူကို အထွက်အဖြစ်ကို အတည်ပြုရန် မေးသည်။ ၎င်းသည် **လုပ်ဆောင်မှုမပြီးနောက်** စီးရီးဖြစ်သည်။ အေးဂျင့်က လုပ်ဆောင်မှုအားဖြင့် ရှေ့ပြေးပြီးဖြစ်သောကြောင့် LLM ခေါ်ဆိုမှု ကုန်ကျစရိတ်ကို အလွန်ပြီးပြီဖြစ်ပြီး၊ ဖြစ်ပေါ်ပြီးသား ဘက်ထွက်သက်ရောက်မှုများမှာ (အီးမေးလ်ပို့ခြင်း၊ ဒေတာဘေ့စ် စာတန်းရေးခြင်း၊ မှတ်ချက်တင်ခြင်း) ဖြစ်ပြီးစီးပါပြီ။

**လုပ်ဆောင်မှုမရှိမီ** စီးရီးသည် ပညာရှင်ခွင့် ရှိုင်ခြင်းမရှိသေးသော အဆင့်ကို ရှေ့မှာ ထည့်သွင်းသည်။ အေးဂျင့်သည် အခြေအနေကို အကြံပြုကာ တံခါးသည် ဆောင်ရွက်ရန်ရှိမရှိ ဆုံးဖြတ်ပြီး အတည်ပြုမှုရရှိမှသာ ဘက်ထွက်သက်ရောက်မှု ဖြစ်ပေါ်သည်။

| အချက် | လုပ်ဆောင်မှုပြီးနောက် အတည်ပြုခြင်း (README ကိုးကားချက်) | လုပ်ဆောင်မှုမရှိမီ တံခါး (ဒီ အမှတ်စု) |
|---|---|---|
| အတည်ပြုမှု ဘယ်အချိန်မှာ ပြုလုပ်သလဲ? | အေးဂျင့် အထွက် ထုတ်လုပ်ပြီးနောက် | ဘယ်ဘက်ထွက်သက်ရောက်မှုမှ မပြုလုပ်ခင် |
| ပယ်ချခံရမှုနဲ့ LLM ကုန်ကျစရိတ် | လုပ်ပြီးပြီ | အကြံပြုစာသာ ကျသင့်၊ လုပ်ဆောင်မှုအပေါ်မရှိ |
| ပြန်လှန်၍မရနိုင်သော ဘက်ထွက်သက်ရောက်မှုများ | ဖြစ်နိုင် (လုပ်ဆောင်မှုပြီးပြီ) | တားဆီးထားသည် |
| စစ်ဆေးမှုပြတ်သားမှု | အတည်ပြုခြင်းသည် သေချာထားသော အမည်ခြင်းဖြစ်သည် | အတည်ပြုခြင်းသည် နှစ်စဉ်၊ လုပ်ဆောင်မှု၊ အကြောင်းရင်းပါဝင်သော JSON မှတ်တမ်းဖြစ်သည် |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## နမူနာ ၂: အန္တရာယ်အဆင့်ခွဲခြားခြင်း

မည်သည့်လုပ်ဆောင်ချက်မျှ လူ့အတည်ပြုမှု မလိုအပ်ပါ။ အသုံးပြုသူထံ အီးမေးလ် ပေးပို့ခြင်းနှင့် public API တစ်ခုသို့ ဖတ်-only lookup တစ်ခုလုပ်ခြင်းတို့သည် စီမံခန့်ခွဲမှု စိုးရိမ်မှုကွာခြားမှုရှိသည်။ ညီမျှစွာ ဆောင်ရွက်ခြင်းသည် စက်တင်တာအာရုံစိုက်မှုကို မသုံးစွဲပဲ ပိုဆွဲခေါ်ပြီး စက်တင်တာအရှိန်ကို နှောင့်နှေးစေသည်။

ရိုးရှင်း သုံးအဆင့် မော်ဒယ်တစ်ခု:

| အဆင့် | ဥပမာများ | အတည်ပြုမှု လည်ပတ်မှု |
|---|---|---|
| `နိမ့်` (ဖတ်-only) | အချက်အလက်အရင်းအမြစ်ရှာဖွေခြင်း၊ ငွေယာယီလေယာဉ်ရွေးချယ်မှုများကြည့်ခြင်း၊ public ဝက်ဘ်စာမျက်နှာများယူခြင်း | အလိုအလျောက် ဆောင်ရွက်ပြီး စစ်ဆေးမှုအတွက် မှတ်တမ်းတင်သည် |
| `အလယ်အလတ်` (စျေးသက်သာသောပြောင်းလဲမှု) | ရလဒ်တစ်ခုကို များစွာ ထားရှိခြင်း၊ ကောင်တာတစ်ခု တိုးမြှင့်ခြင်း၊ သတိပေးချက် တစ်ခုကို အချိန်ဇယားဖွဲ့ခြင်း | အလိုအလျောက် ဆောင်ရွက်သည်၊ ဒါပေမယ့် နေ့စဉ် ပုံမှန် သုံးသပ်ပြန်လည်ကြည့်ရှုမှုဖြင့် ဖြစ်သည် |
| `အမြင့်` (အပြင်ဘက်ကို သက်ဆိုင်သော သို့မဟုတ် ပြောင်းလဲ၍မရသော) | အီးမေးလ် ပေးပို့ခြင်း၊ ကတ်မှ ငွေခြင်း၊ အများပြည်သူ ချန်နယ်တွင် ပို့ခြင်း | လူ့အတည်ပြုမှုအတွက် တားမြစ်ထားသည် |

၎င်းသည် tiering တစ်ခုသာဖြစ်သည်။ ထုတ်လုပ်မှု စနစ်များသည် ပို၍ ဂရမ်မာတိကျသော အဆင့်များ အသုံးပြုလေ့ ရှိသည် (ဥပမာ- AWS IAM ခွင့်ပြုချက်အဆင့်များ၊ အခန်းကဏ္ဍအခြေခံ ဝင်ခွင့်အဆင့်များ)။ အောက်ပါ သုံးအဆင့် မော်ဒယ်သည် ဖတ်-only နှင့် နောက်ဆက်တွဲ လုပ်ဆောင်ချက်များကို ပေါင်းစပ်ထားသော လုပ်ငန်းရှင်အတွက် အသုံးဝင်သော အနည်းဆုံး မော်ဒယ် ဖြစ်သည်။

အောက်ပါ ရှာဖွေမှုသည် မေရ။ဟာမစ် ဆဲလlections ကို အသုံးပြုထားပြီး သရုပ်ပြကို မပြောင်းလဲပဲစျေးသက်သာစေသည်။ ထုတ်လုပ်မှု စနစ်တစ်ခုတွင် သင်သည် ၎င်းကို သင်ယူထားသော ချပြသည့် အဆင့်သတ်မှတ်စနစ် သို့မဟုတ် မူဝါဒ အင်ဂျင်ဖြင့် ပြောင်းလဲအသုံးပြုမည် ဖြစ်သည်။


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## ပုံစံ ၃: စစ်ဆေးမှတ်တမ်းနှင့် ပြန်လည်သုံးသပ်မှု အကြိမ်ကြိမ် လည်ပတ်မှု

`print("Response approved.")` ဆိုတာ စစ်ဆေးမှတ်တမ်း မဟုတ်ပါ။ ယုံကြည်ချက်အတွက် Gate ဆုံးဖြတ်ချက်တိုင်းကို သင်နောက်ပိုင်းတွင် မေးခွန်းမေး၊ ပြန်လည်သွားရှေ့ကောက်နိုင်ရန် သို့မဟုတ် ဖြစ်ရပ် သုံးသပ်မှုတွင် ဖော်ပြနိုင်ရန် စနစ်တကျ ဖြစ်အောင် မှတ်တမ်းတင်ထားသင့်သည်။

အပိုင်း ၂ ခုရှိသည်။

၁။ **Append-only JSONL။** ဆုံးဖြတ်ချက်တစ်ခုစီအတွက် တန်းတစ်ကြောင်း၊ အချိန်ကာလ၊ လုပ်ဆောင်ချက်၊ တန်းအဆင့်၊ ဆုံးဖြတ်ချက်နှင့် ကြောင်းပြချက်ပါဝင်သည်။ grepရိုက်ရန် လွယ်ကူပြီး နောက်ပိုင်းတွင် အမှန်တကယ် မှတ်တမ်းသိုလှောင်ရာသို့ ပို့ရလွယ်ကူသည်။
၂။ **ငြင်းပယ်မှုဖြင့် ပြန်လည်သုံးသပ်မှု လည်ပတ်မှု။** Gate က `deny` ပြန်လာသောအခါ agent သည် ငြင်းပယ်ခြင်းကြောင်းပြချက်ကို နောက်ခံအချက်အလက်အဖြစ် ကိုယ်တိုင်ထပ်မံ မေးမြန်းခြင်းဖြင့် နောက်ထပ် အကြံပြုချက်သည် ပြဿနာကိုရှောင်ကြဉ်နိုင်စေရန် ဖြစ်သည်။


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## ထပ်ဆောင်း အရင်းအမြစ်များ  

အများပြည်သူအသုံးပြု နောက်ထပ်ပရောဂျက်များအချို့က ဒီ HITL ပုံစံမျိုးစုံကို အကောင်အထည်ဖော်ထားကြသည်။ မိမိစက်ရုံအတွက် ကိုက်ညီသည့်နည်းလမ်းကို သုံးထုံးစမ်းကြည့်ရန် ဖော်ပြချက်များကို နှိုင်းယှဥ်ပါ။  

- **LangChain** human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): လူ့အာရုံလိုအပ်ချက်အတွက် တားမြစ်ထားသော အကိရိယာ wrapper များဖြစ်ပြီး အလုပ်လုပ်စဉ်ကို ရပ်ဆိုင်းစေသည်။  
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ပြုပြင်ပြောင်းလဲခဲ့ပါသည်): လူများကို ကိုယ်စားပြု Multi-agent တွေကြားစာနာခြင်းအတွက် agent အခန်းကဏ္ဍတစ်ခု သတ်မှတ်ထားသည်။  
- **Microsoft Agent Framework (MAF)** function-invocation middleware ([docs](https://learn.microsoft.com/agent-framework/)): tool/function ဖုန်းခေါ်မှုတိုင်းအတွက် လည်ပတ်သည့် middleware ဖြစ်ပြီး gating logic နဲ့ အတည်ပြုမှုစနစ်များအတွက်သင့်လျော်သည်။  

ပရောဂျက်တိုင်းက ဒီ သုံးမျိုး subconcept များကို မတူညီစွာ ကိုင်တွယ်ထားသည်။ LangChain က tool အဖြစ် wrap လုပ်ပြီး၊ AutoGen က agent အခန်းကဏ္ဍ အသုံးပြုသည်။ Microsoft Agent Framework က function-invocation middleware အသုံးပြုသည်။ မိမိတို့၏ agent ဒီဇိုင်းကို ရွေးချယ်ရန်မှီ မိနစ်တစ်နှစ်ချိန် အကောင်အထည်ဖော်မှုတစ်ခု ဒါမှမဟုတ် နှစ်ခုကို စတင်အဆင့်တိုင်း လေ့လာဖတ်ရှုပါ။  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
